In [ ]:
import os, json, subprocess, sys, time, glob, shutil, urllib.request
CATEGORIES = 'bugfix,feature'
print('python', sys.version.split()[0], '| categories:', CATEGORIES)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:1000])
gpu = subprocess.run(['nvidia-smi', '--query-gpu=memory.total,name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip().splitlines()
total_mem = sum(int(line.split()[0].replace(',', '')) for line in gpu if line)
print('GPU count:', len(gpu), '| total VRAM MiB:', total_mem)


In [ ]:
os.chdir('/kaggle/working')
r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/mattdani21/ModelSwapper.git'], capture_output=True, text=True)
print(r.stdout[-300:], r.stderr[-300:])
os.chdir('/kaggle/working/ModelSwapper')
print(subprocess.run(['git', 'log', '-1', '--format=%h %ci'], capture_output=True, text=True).stdout)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'pytest'], capture_output=True, text=True)
print('pytest install rc:', r.returncode)


In [ ]:
# CUDA layout on Kaggle (diag-verified): toolkit 12.8, driver lib ONLY in
# /usr/local/cuda-12.8/compat — symlink into toolkit lib64 for CMake.
COMPAT = '/usr/local/cuda-12.8/compat'
for lib in ['libcuda.so', 'libcuda.so.1']:
    src = COMPAT + '/' + lib
    if os.path.exists(src):
        subprocess.run(['bash', '-c', 'ln -sf ' + src + ' /usr/local/cuda/lib64/' + lib], capture_output=True, text=True)
        print('symlinked', src)
os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/working/llama.cpp/build/bin/llama-server'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git'], capture_output=True, text=True, check=True)
    r = subprocess.run(['cmake', '-B', '/kaggle/working/llama.cpp/build', '-S', '/kaggle/working/llama.cpp',
                        '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release',
                        '-DCUDAToolkit_ROOT=/usr/local/cuda-12.8',
                        '-DCMAKE_LIBRARY_PATH=' + COMPAT + ':/usr/local/cuda/lib64'],
                       capture_output=True, text=True, timeout=900)
    print('cmake configure rc:', r.returncode)
    if r.returncode != 0:
        print(r.stdout[-1500:])
        print(r.stderr[-1500:])
    nproc = subprocess.run(['nproc'], capture_output=True, text=True).stdout.strip() or '4'
    r = subprocess.run(['cmake', '--build', '/kaggle/working/llama.cpp/build', '--config', 'Release',
                        '-j', nproc, '--target', 'llama-server'], capture_output=True, text=True, timeout=3600)
    print('cmake build rc:', r.returncode)
    if r.returncode != 0:
        print(r.stdout[-1500:])
        print(r.stderr[-1500:])
bin_path = '/kaggle/working/llama.cpp/build/bin/llama-server'
assert os.path.exists(bin_path), 'llama-server build failed'
# Trim build footprint: keep only the binary (models live on the input
# mount; /kaggle/working quota ~20GB cannot hold build + 23.7GB of GGUFs).
subprocess.run(['cp', bin_path, '/kaggle/working/llama-server'], check=True)
subprocess.run(['rm', '-rf', '/kaggle/working/llama.cpp'])
os.environ['LLAMA_SERVER'] = '/kaggle/working/llama-server'
os.environ['PATH'] = '/kaggle/working:' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = COMPAT + ':' + os.environ.get('LD_LIBRARY_PATH', '')
print('llama-server ready')


In [ ]:
# Models stay on the /kaggle/input mount (quota check + read-speed probe).
# The v4 run's real time sink was Qwen3 thinking tokens, not loads — with
# thinking disabled the remaining load cost (~1-2 min/task) is acceptable.
print(subprocess.run(['df', '-h', '/kaggle/working'], capture_output=True, text=True).stdout)
t0 = time.time()
r = subprocess.run(['dd', 'if=/kaggle/input/swapos-ggufs/Qwen3-14B-Q4_K_M.gguf', 'of=/dev/null', 'bs=1M', 'count=2048'], capture_output=True, text=True)
print('input read probe (2GB):', round(time.time() - t0), 's')
MODEL_PATHS = {
    'reason': '/kaggle/input/swapos-ggufs/Qwen3-14B-Q4_K_M.gguf',
    'code': '/kaggle/input/swapos-ggufs/Qwen3-Coder-30B-A3B-Instruct-Q3_K_M.gguf',
    'review': '/kaggle/input/swapos-ggufs/Qwen3-14B-Q4_K_M.gguf',
}


In [ ]:
os.chdir('/kaggle/working/ModelSwapper')
env = dict(os.environ)
env['LLAMA_CONTEXT'] = '4096'
env['LLAMA_NGPU'] = '99'
cmd = [sys.executable, 'pipeline/run_pipeline.py',
       '--models-json', json.dumps(MODEL_PATHS),
       '--out', '/kaggle/working/pipeline-results.json',
       '--capsule-dir', '/kaggle/working/capsules',
       '--categories', CATEGORIES,
       '--port-base', '8950',
       '--max-iterations', '3',
       '--max-tokens', '2048']
print('running pipeline eval...')
t0 = time.time()
try:
    r = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=8 * 3600)
    print('pipeline rc:', r.returncode, '| wall:', round((time.time() - t0) / 60, 1), 'min')
    print((r.stdout or '')[-2500:])
    print((r.stderr or '')[-1000:])
except subprocess.TimeoutExpired:
    print('PIPELINE TIMEOUT after', round((time.time() - t0) / 60, 1), 'min — partial results kept')


In [ ]:
os.chdir('/kaggle/working')
res_path = '/kaggle/working/pipeline-results.json'
if os.path.exists(res_path):
    d = json.load(open(res_path))
    print('PASS RATE:', d.get('pass_rate'), '|', d.get('tasks_passed'), '/', d.get('tasks_total'))
    print('per_category:', d.get('per_category'))
    print('mean_wall_clock_s:', d.get('mean_wall_clock_s'))
    print('mean_load_s:', d.get('mean_load_s'), '| mean_evict_s:', d.get('mean_evict_s'))
    with open('/kaggle/working/pipeline-summary.txt', 'w') as f:
        f.write(json.dumps({k: v for k, v in d.items() if k != 'results'}, indent=2))
else:
    print('NO RESULTS FILE')
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working', 'pipeline-results.json')
shutil.make_archive('/kaggle/working/capsules', 'zip', '/kaggle/working/capsules')
print('outputs:', sorted(glob.glob('/kaggle/working/results*') + glob.glob('/kaggle/working/capsules*')))
